# trust4 & PanPep

In [ ]:
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict

def collapse_trust4_by_cdr3aa(trust4_df):
    """
     CDR3aa TRUST4results
    - count( )
    - record with CDR3nt
    - V/D/J/C, count record annotation
    - allCDR3ntsequenceinformation
    """
    collapsed_data = defaultdict(lambda: {
        'records': []
    })
    # all record
    for _, row in trust4_df.iterrows():
        cdr3aa = row['CDR3aa']
        collapsed_data[cdr3aa]['records'].append(row.to_dict())
        # processeachCDR3aa
        result_rows = []
        for cdr3aa, data in collapsed_data.items():
            records = data['records']
            n_variants = len(records)
            # calculate count and frequency
            total_count = sum(rec['#count'] for rec in records)
            total_frequency = sum(rec['frequency'] for rec in records)
            # count, count record for annotation
            records_sorted = sorted(records, key=lambda x: x['#count'], reverse=True)
            primary_record = records_sorted[0]
            # allCDR3ntsequenceand count
            cdr3nt_info = [f"{rec['CDR3nt']}({rec['#count']})" for rec in records_sorted]
            # check with annotation
            unique_annotations = {
                'V': set(str(rec['V']) for rec in records),
                'D': set(str(rec['D']) for rec in records),
                'J': set(str(rec['J']) for rec in records),
                'C': set(str(rec['C']) for rec in records),
                'cid': set(str(rec['cid']) for rec in records),
                'cid_full_length': set(str(rec['cid_full_length']) for rec in records)
            }
            has_annotation_conflict = any(len(vals) > 1 for vals in unique_annotations.values())
            # buildresultsrows
            row_data = {
                'CDR3aa': cdr3aa,
                'total_count': total_count, # count
                'total_frequency': total_frequency, # frequency
                'n_variants': n_variants, # CDR3nt count
                'CDR3nt_with_counts': ';'.join(cdr3nt_info), # allCDR3nt count
                # count record annotation
                'V': primary_record['V'],
                'D': primary_record['D'],
                'J': primary_record['J'],
                'C': primary_record['C'],
                'cid': primary_record['cid'],
                'cid_full_length': primary_record['cid_full_length'],
                'primary_variant_count': primary_record['#count'], # count
                'has_annotation_conflict': has_annotation_conflict
            }
            # with coverage, all information
            if has_annotation_conflict:
                for col, vals in unique_annotations.items():
                    if len(vals) > 1:
                        row_data[f'{col}_all_variants'] = ';'.join(sorted(vals))
                        result_rows.append(row_data)
                        return pd.DataFrame(result_rows)

def merge_trust4_with_panpep(trust4_file, panpep_file, output_file):
    """
     TRUST4results andPanPep results
    """
    sample_name = os.path.basename(panpep_file).split('.')[2]
    mhc_type = os.path.basename(panpep_file).split('.')[3]
    print(f"\n{'-'*60}")
    print(f"Sample: {sample_name} | MHC Type: {mhc_type}")
    print(f"{'-'*60}")
    # readTRUST4file
    if not os.path.exists(trust4_file):
        print(f"Failed TRUST4 file not found: {trust4_file}")
        return False
    try:
        trust4_df = pd.read_csv(trust4_file, sep='\t')
        print(f"Statistics: TRUST4: {len(trust4_df)} records")
    except Exception as e:
        print(f"Failed Error reading TRUST4: {e}")
        return False
    # readPanPepfile
    if not os.path.exists(panpep_file):
        print(f"Failed PanPep file not found: {panpep_file}")
        return False
    try:
        panpep_df = pd.read_csv(panpep_file)
        print(f"Statistics: PanPep: {len(panpep_df)} records")
        print(f" Unique CDR3: {panpep_df['CDR3'].nunique()}")
    except Exception as e:
        print(f"Failed Error reading PanPep: {e}")
        return False
    # TRUST4data
    print(f"Processing: Collapsing TRUST4 by CDR3aa...")
    collapsed_trust4 = collapse_trust4_by_cdr3aa(trust4_df)
    # statistics
    n_unique_cdr3aa = len(collapsed_trust4)
    n_multi_variant = (collapsed_trust4['n_variants'] > 1).sum()
    n_with_conflict = collapsed_trust4['has_annotation_conflict'].sum()
    print(f" -> {n_unique_cdr3aa} unique CDR3aa")
    print(f" -> {n_multi_variant} have multiple CDR3nt variants")
    if n_with_conflict > 0:
        print(f" Warning {n_with_conflict} have annotation conflicts")
        # data
        print(f"Merge: Merging with PanPep predictions...")
        merged_df = panpep_df.merge(
            collapsed_trust4,
            left_on='CDR3',
            right_on='CDR3aa',
            how='left'
)
# column column
priority_cols = [
    'Peptide', 'CDR3', 'Score', # PanPep column
    'total_count', 'total_frequency', # information
    'n_variants', 'primary_variant_count', # information
    'CDR3nt_with_counts', # detailedCDR3ntinformation
    'V', 'D', 'J', 'C', 'cid', 'cid_full_length', # annotationinformation
    'has_annotation_conflict' #
]
# does not exist column
existing_priority_cols = [c for c in priority_cols if c in merged_df.columns]
# column( *_all_variants)
other_cols = [c for c in merged_df.columns if c not in existing_priority_cols and c != 'CDR3aa']
merged_df = merged_df[existing_priority_cols + other_cols]
# statistics matched
matched = merged_df['total_count'].notna().sum()
unmatched = merged_df['total_count'].isna().sum()
print(f"Summary: Matching results:")
print(f" OK Matched: {matched} ({matched/len(merged_df)*100:.1f}%)")
if unmatched > 0:
    print(f" Warning Unmatched: {unmatched} ({unmatched/len(merged_df)*100:.1f}%)")
    # count statistics
    if matched > 0:
        matched_data = merged_df[merged_df['total_count'].notna()]
        print(f"Statistics: Count statistics (matched CDR3):")
        print(f" Total count sum: {matched_data['total_count'].sum():.0f}")
        print(f" Mean count: {matched_data['total_count'].mean():.1f}")
        print(f" Median count: {matched_data['total_count'].median():.0f}")
        # with coverage
        multi_variant = matched_data[matched_data['n_variants'] > 1]
        if len(multi_variant) > 0:
            print(f" Multi-variant CDR3: {len(multi_variant)}")
            print(f" Max variants: {multi_variant['n_variants'].max()}")
            # saveresults
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            merged_df.to_csv(output_file, index=False)
            print(f"OK Saved: {os.path.basename(output_file)}")
            return True

def generate_summary_report(base_results_dir):
    """
    generatereport
    """
    projects = ['CRC', 'CRC_MS']
    all_stats = []
    for project in projects:
        merged_dir = os.path.join(base_results_dir, project, '07.TRUST4', 'PanPep_merged')
        if not os.path.exists(merged_dir):
            continue
        for file in os.listdir(merged_dir):
            if not file.endswith('_merged.csv'):
                continue
            filepath = os.path.join(merged_dir, file)
            try:
                df = pd.read_csv(filepath)
                parts = file.replace('_merged.csv', '').split('.')
                sample = parts[2]
                mhc_type = parts[3]
                matched = df['total_count'].notna().sum()
                total = len(df)
                multi_variant = (df['n_variants'] > 1).sum() if 'n_variants' in df.columns else 0
                all_stats.append({
                    'Project': project,
                    'Sample': sample,
                    'MHC_Type': mhc_type,
                    'Total_Predictions': total,
                    'Matched': matched,
                    'Match_Rate': f"{matched/total*100:.1f}%",
                    'Multi_Variant_CDR3': multi_variant
                })
            except Exception as e:
                print(f"Error reading {file}: {e}")
                return pd.DataFrame(all_stats)

def process_all_samples(base_data_dir, base_results_dir):
    """
    process all samples
    """
    projects = {
        'CRC': {'group': 'R'},
        'CRC_MS': {'group': 'T'}
    }
    success_count = 0
    for project, config in projects.items():
        print(f"\n{'='*60}")
        print(f"Project: PROJECT: {project}")
        print(f"{'='*60}")
        group = config['group']
        trust4_base = os.path.join(base_data_dir, project, group, 'TRUST4')
        panpep_dir = os.path.join(base_results_dir, project, '07.TRUST4', 'PanPep')
        output_dir = os.path.join(base_results_dir, project, '07.TRUST4', 'PanPep_merged')
        if not os.path.exists(panpep_dir):
            print(f"Failed PanPep directory not found: {panpep_dir}")
            continue
        # filenameformat: _v2.csv and .csv
        panpep_files = sorted([f for f in os.listdir(panpep_dir) if f.endswith('.panpep_zero_shot_out_v2.csv') or f.endswith('.panpep_zero_shot_out.csv')])
        print(f"Files: Found {len(panpep_files)} PanPep files")
        for panpep_file in panpep_files:
            # fileafter
            if panpep_file.endswith('_v2.csv'):
                parts = panpep_file.replace('.panpep_zero_shot_out_v2.csv', '').split('.')
            else:
                parts = panpep_file.replace('.panpep_zero_shot_out.csv', '').split('.')
                if len(parts) >= 4:
                    sample_name = parts[2]
                else:
                    print(f"Warning Cannot parse: {panpep_file}")
                    continue
                trust4_file = os.path.join(
                    trust4_base, sample_name, '01.trust4',
                    f'TRUST_{sample_name}_report.tsv'
)
output_file = os.path.join(
    output_dir,
    panpep_file.replace('.csv', '_merged.csv')
)
if merge_trust4_with_panpep(trust4_file, os.path.join(panpep_dir, panpep_file),
    output_file):
success_count += 1
return success_count

if __name__ == '__main__':
    BASE_DATA_DIR = '/path/to/project/data_V3/special'
    BASE_RESULTS_DIR = '/path/to/project/results_V3/special'
    print("\n" + "="*60)
    print("Tool: TRUST4 & PanPep Integration Tool")
    print("="*60)
    print(f"Folder: Data: {BASE_DATA_DIR}")
    print(f"Folder: Results: {BASE_RESULTS_DIR}")
    print("="*60)
    print("\nTarget: Strategy:")
    print(" - Accumulate count for same CDR3aa")
    print(" - Track number of CDR3nt variants")
    print(" - Use highest-count variant for annotations")
    print(" - Flag annotation conflicts when present")
    print("="*60)
    # process all samples
    n_processed = process_all_samples(BASE_DATA_DIR, BASE_RESULTS_DIR)
    # generatereport
    print("\n" + "="*60)
    print("Statistics: Generating Summary Report")
    print("="*60)
    summary_df = generate_summary_report(BASE_RESULTS_DIR)
    if len(summary_df) > 0:
        summary_file = os.path.join(BASE_RESULTS_DIR, 'merge_summary.csv')
        summary_df.to_csv(summary_file, index=False)
        print(f"\n{summary_df.to_string(index=False)}")
        print(f"\nOK Summary saved to: {summary_file}")
        print("\n" + "="*60)
        print(f"Completed: Completed! Processed {n_processed} files")
        print("="*60)

# merge MTR peptide

In [ ]:
import pandas as pd
import os
from pathlib import Path

def load_filtered_peptides(filtered_file):
    """
    readfilteredfile, extractpeptide MHCinformation
    return DataFrame, peptide, MHC, Affinity(nM), BindLevel
    Summary: Class Ifilein and column namesfor Aff(nM), Class IIfor Affinity(nM)
    """
    if not os.path.exists(filtered_file):
        return None
    try:
        df = pd.read_csv(filtered_file, sep='\t')
        # checkrequired columns - and column names Aff(nM) or Affinity(nM)
        base_required_cols = ['peptide', 'MHC', 'BindLevel']
        # and column names
        affinity_col = None
        if 'Aff(nM)' in df.columns:
            affinity_col = 'Aff(nM)'
        elif 'Affinity(nM)' in df.columns:
            affinity_col = 'Affinity(nM)'
        else:
            print(f" Warning No affinity column found (expected 'Aff(nM)' or 'Affinity(nM)')")
            return None
        required_cols = base_required_cols + [affinity_col]
        # check all columndoes not exist
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f" Warning Missing columns in filtered file: {missing_cols}")
            return None
        # extract column
        peptide_info = df[required_cols].copy()
        # column namesfor Affinity(nM)
        if affinity_col == 'Aff(nM)':
            peptide_info = peptide_info.rename(columns={'Aff(nM)': 'Affinity(nM)'})
            # deduplicate - peptide with MHCrecord, and (Affinity )
            peptide_info = peptide_info.sort_values('Affinity(nM)')
            peptide_info = peptide_info.drop_duplicates(subset='peptide', keep='first')
            return peptide_info
    except Exception as e:
        print(f" Failed Error reading filtered file: {e}")
        return None

def add_mhc_info_to_merged(merged_file, filtered_info, output_file):
    """
     MHCinformation tomergedfilein
    Summary:
    merged_file: firstgeneratemergedfilepath
    filtered_info: peptide MHCinformationDataFrame
    output_file: Output filespath
    """
    if not os.path.exists(merged_file):
        print(f" Failed Merged file not found: {merged_file}")
        return False
    if filtered_info is None or len(filtered_info) == 0:
        print(f" Warning No filtered peptide info available")
        return False
    try:
        # readmergedfile
        merged_df = pd.read_csv(merged_file)
        original_len = len(merged_df)
        print(f" Statistics: Merged file: {original_len} records")
        print(f" Statistics: Filtered peptides: {len(filtered_info)} unique peptides")
        # MHCinformation
        # left join mergedfileinallrecord
        enhanced_df = merged_df.merge(
            filtered_info,
            left_on='Peptide',
            right_on='peptide',
            how='left',
            suffixes=('', '_filtered')
)
# peptide column( filtered_info)
if 'peptide' in enhanced_df.columns:
    enhanced_df = enhanced_df.drop(columns=['peptide'])
    # column column, MHCinformation first
    priority_cols = [
        'Peptide', 'CDR3', 'Score', # PanPep column
        'MHC', 'Affinity(nM)', 'BindLevel', # MHCinformation
        'total_count', 'total_frequency', # TRUST4 information
        'n_variants', 'primary_variant_count',
        'CDR3nt_with_counts',
        'V', 'D', 'J', 'C', 'cid', 'cid_full_length',
        'has_annotation_conflict'
    ]
    existing_priority_cols = [c for c in priority_cols if c in enhanced_df.columns]
    other_cols = [c for c in enhanced_df.columns if c not in existing_priority_cols]
    enhanced_df = enhanced_df[existing_priority_cols + other_cols]
    # statistics matched
    n_with_mhc = enhanced_df['MHC'].notna().sum()
    n_without_mhc = enhanced_df['MHC'].isna().sum()
    print(f" Summary: MHC info added:")
    print(f" OK With MHC info: {n_with_mhc} ({n_with_mhc/original_len*100:.1f}%)")
    print(f" - Without MHC info: {n_without_mhc} ({n_without_mhc/original_len*100:.1f}%)")
    # BindLevel ( withMHCinformationrecord)
    if n_with_mhc > 0:
        bind_level_counts = enhanced_df[enhanced_df['MHC'].notna()]['BindLevel'].value_counts()
        print(f" Target: BindLevel distribution:")
        for level, count in bind_level_counts.items():
            print(f" {level}: {count}")
            # saveresults
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            enhanced_df.to_csv(output_file, index=False)
            print(f" OK Saved: {os.path.basename(output_file)}")
            return True
            except Exception as e:
                print(f" Failed Error processing: {e}")
                return False

def process_single_sample(project, group, sample, mhc_class, base_results_dir):
    """
    process samples MHC class
    Summary:
    project: 'CRC' or 'CRC_MS'
    group: 'R' or 'T'
    sample: sample
    mhc_class: 'I' or 'II'
    base_results_dir: results directory
    """
    print(f"\n{'-'*60}")
    print(f"Sample: {sample} | Class {mhc_class}")
    print(f"{'-'*60}")
    # buildfilepath
    # Filteredfilepath
    filtered_file = os.path.join(
        base_results_dir, project,
        '06.antigen', '06.2.MTR_peptides', 'R_fully_contained',
        f'{sample}_class{mhc_class}_filtered.tsv'
)
# Mergedfilepath - filenameformat
if mhc_class == 'I':
    mhc_suffix = 'p_I'
else:
    mhc_suffix = 'p_II'
    # filenameformat
    merged_file_v2 = os.path.join(
        base_results_dir, project,
        '07.TRUST4', 'PanPep_merged',
        f'{project}.{group}.{sample}.{mhc_suffix}.panpep_zero_shot_out_v2_merged.csv'
)
merged_file_v1 = os.path.join(
    base_results_dir, project,
    '07.TRUST4', 'PanPep_merged',
    f'{project}.{group}.{sample}.{mhc_suffix}.panpep_zero_shot_out_merged.csv'
)
# does not existfile
if os.path.exists(merged_file_v2):
    merged_file = merged_file_v2
elif os.path.exists(merged_file_v1):
    merged_file = merged_file_v1
else:
    print(f" Failed Merged file not found (tried both v1 and v2 formats)")
    return False
    # Output filespath
    output_dir = os.path.join(
        base_results_dir, project,
        '07.TRUST4', 'PanPep_merged_with_MHC'
)
# file buildoutputfilename
base_name = os.path.basename(merged_file).replace('_merged.csv', '')
output_file = os.path.join(output_dir, f'{base_name}_merged_with_MHC.csv')
print(f"Folder: Filtered file: {os.path.basename(filtered_file)}")
print(f"Folder: Merged file: {os.path.basename(merged_file)}")
# readfiltered peptideinformation
filtered_info = load_filtered_peptides(filtered_file)
# MHCinformationtomergedfile
success = add_mhc_info_to_merged(merged_file, filtered_info, output_file)
return success

def process_all_samples(base_results_dir):
    """
    process all samples
    """
    projects = {
        'CRC': {'group': 'R'},
        'CRC_MS': {'group': 'T'}
    }
    success_count = 0
    total_count = 0
    for project, config in projects.items():
        print(f"\n{'='*60}")
        print(f"Project: PROJECT: {project}")
        print(f"{'='*60}")
        group = config['group']
        # allmergedfile
        merged_dir = os.path.join(
            base_results_dir, project,
            '07.TRUST4', 'PanPep_merged'
)
if not os.path.exists(merged_dir):
    print(f"Failed Merged directory not found: {merged_dir}")
    continue
    merged_files = sorted([f for f in os.listdir(merged_dir)
        if f.endswith('_merged.csv') and ('panpep_zero_shot_out_v2_merged.csv' in f or 'panpep_zero_shot_out_merged.csv' in f)])
    print(f"Files: Found {len(merged_files)} merged files")
    for merged_file in merged_files:
        # filename - format
        if '_v2_merged.csv' in merged_file:
            parts = merged_file.replace('.panpep_zero_shot_out_v2_merged.csv', '').split('.')
        else:
            parts = merged_file.replace('.panpep_zero_shot_out_merged.csv', '').split('.')
            if len(parts) >= 4:
                sample = parts[2]
                mhc_suffix = parts[3] # p_I or p_II
                # forclass I or class II
                if mhc_suffix == 'p_I':
                    mhc_class = 'I'
                elif mhc_suffix == 'p_II':
                    mhc_class = 'II'
                else:
                    print(f"Warning Unknown MHC suffix: {mhc_suffix}")
                    continue
                total_count += 1
                if process_single_sample(project, group, sample, mhc_class, base_results_dir):
                    success_count += 1
                else:
                    print(f"Warning Cannot parse filename: {merged_file}")
                    return success_count, total_count

def generate_summary_report(base_results_dir):
    """
    generateafterdata report
    """
    projects = ['CRC', 'CRC_MS']
    all_stats = []
    for project in projects:
        enhanced_dir = os.path.join(
            base_results_dir, project,
            '07.TRUST4', 'PanPep_merged_with_MHC'
)
if not os.path.exists(enhanced_dir):
    continue
    for file in os.listdir(enhanced_dir):
        if not file.endswith('_with_MHC.csv'):
            continue
        filepath = os.path.join(enhanced_dir, file)
        try:
            df = pd.read_csv(filepath)
            # filename - format
            if '_v2_merged_with_MHC.csv' in file:
                parts = file.replace('_v2_merged_with_MHC.csv', '').split('.')
            else:
                parts = file.replace('_merged_with_MHC.csv', '').split('.')
                if len(parts) < 3:
                    continue
                sample = parts[2]
                mhc_type = 'Class I' if 'p_I' in file else 'Class II'
                total = len(df)
                with_mhc = df['MHC'].notna().sum()
                # statisticsBindLevel
                bind_levels = {}
                if with_mhc > 0:
                    for level in ['<= SB', '<= WB', 'No bind']:
                        count = (df['BindLevel'] == level).sum()
                        if count > 0:
                            bind_levels[level] = count
                            stats = {
                                'Project': project,
                                'Sample': sample,
                                'MHC_Class': mhc_type,
                                'Total_Records': total,
                                'With_MHC_Info': with_mhc,
                                'MHC_Coverage': f"{with_mhc/total*100:.1f}%"
                            }
                            # BindLevelstatistics
                            for level, count in bind_levels.items():
                                stats[level] = count
                                all_stats.append(stats)
        except Exception as e:
            print(f"Error reading {file}: {e}")
            return pd.DataFrame(all_stats)

if __name__ == '__main__':
    BASE_RESULTS_DIR = '/path/to/project/results_V3/special'
    print("\n" + "="*60)
    print("Tool: Add MHC Information Tool")
    print("="*60)
    print(f"Folder: Results directory: {BASE_RESULTS_DIR}")
    print("="*60)
    print("\nTarget: Task:")
    print(" - Extract MHC, Affinity, BindLevel from filtered files")
    print(" - Add to merged PanPep-TRUST4 results")
    print(" - Match by Peptide sequence")
    print("="*60)
    # process all samples
    success, total = process_all_samples(BASE_RESULTS_DIR)
    # generatereport
    print("\n" + "="*60)
    print("Statistics: Generating Summary Report")
    print("="*60)
    summary_df = generate_summary_report(BASE_RESULTS_DIR)
    if len(summary_df) > 0:
        summary_file = os.path.join(BASE_RESULTS_DIR, 'mhc_info_summary.csv')
        summary_df.to_csv(summary_file, index=False)
        print(f"\n{summary_df.to_string(index=False)}")
        print(f"\nOK Summary saved to: {summary_file}")
        print("\n" + "="*60)
        print(f"Completed: Completed! Processed {success}/{total} files successfully")
        print("="*60)
        print("\nFiles: Output directory:")
        print(" */07.TRUST4/PanPep_merged_with_MHC/")
        print("="*60)

In [ ]:
import pandas as pd
import os
from pathlib import Path

def load_filtered_peptides(filtered_file):
    """
    readfilteredfile, extractpeptide MHCinformation
    return DataFrame, peptide, MHC, Affinity(nM), BindLevel
     returnall peptide list( )
    Summary: Class Ifilein and column namesfor Aff(nM), Class IIfor Affinity(nM)
    """
    if not os.path.exists(filtered_file):
        return None, None
    try:
        df = pd.read_csv(filtered_file, sep='\t')
        # extractall peptide( BindLevel )
        if 'peptide' not in df.columns:
            print(f" Warning 'peptide' column not found in filtered file")
            return None, None
        all_filtered_peptides = set(df['peptide'].dropna().unique())
        print(f" Statistics: Total filtered peptides: {len(all_filtered_peptides)}")
        # checkrequired columns - and column names Aff(nM) or Affinity(nM)
        base_required_cols = ['peptide', 'MHC']
        # and column names
        affinity_col = None
        if 'Aff(nM)' in df.columns:
            affinity_col = 'Aff(nM)'
        elif 'Affinity(nM)' in df.columns:
            affinity_col = 'Affinity(nM)'
        else:
            print(f" Warning No affinity column found (expected 'Aff(nM)' or 'Affinity(nM)')")
            # returnpeptidelist
            return None, all_filtered_peptides
        # BindLevel for, for column
        required_cols = base_required_cols + [affinity_col]
        if 'BindLevel' in df.columns:
            required_cols.append('BindLevel')
            # extract column
            peptide_info = df[required_cols].copy()
            # column namesfor Affinity(nM)
            if affinity_col == 'Aff(nM)':
                peptide_info = peptide_info.rename(columns={'Aff(nM)': 'Affinity(nM)'})
                # deduplicate - peptide with MHCrecord, and (Affinity )
                peptide_info = peptide_info.sort_values('Affinity(nM)')
                peptide_info = peptide_info.drop_duplicates(subset='peptide', keep='first')
                # statisticsBindLevel
                if 'BindLevel' in peptide_info.columns:
                    non_empty_bind = peptide_info['BindLevel'].notna().sum()
                    empty_bind = peptide_info['BindLevel'].isna().sum()
                    print(f" With BindLevel: {non_empty_bind}, Empty BindLevel: {empty_bind}")
                    return peptide_info, all_filtered_peptides
    except Exception as e:
        print(f" Failed Error reading filtered file: {e}")
        return None, None

def add_mhc_info_to_merged(merged_file, filtered_info, filtered_peptides, output_file):
    """
     MHCinformation tomergedfilein
    Summary:
    merged_file: firstgeneratemergedfilepath
    filtered_info: peptide MHCinformationDataFrame
    filtered_peptides: all peptide ( )
    output_file: Output filespath
    """
    if not os.path.exists(merged_file):
        print(f" Failed Merged file not found: {merged_file}")
        return False
    try:
        # readmergedfile
        merged_df = pd.read_csv(merged_file)
        original_len = len(merged_df)
        print(f" Statistics: Merged file: {original_len} records")
        # all peptide for peptide
        if filtered_peptides is not None:
            merged_df['is_filtered_peptide'] = merged_df['Peptide'].isin(filtered_peptides)
            n_of_interest = merged_df['is_filtered_peptide'].sum()
            print(f" Target: Peptides of interest: {n_of_interest} records ({n_of_interest/original_len*100:.1f}%)")
        else:
            merged_df['is_filtered_peptide'] = False
            print(f" Warning No filtered peptide list available")
            # withMHCinformation,
            if filtered_info is not None and len(filtered_info) > 0:
                print(f" Statistics: Filtered peptide info: {len(filtered_info)} unique peptides")
                # MHCinformation
                enhanced_df = merged_df.merge(
                    filtered_info,
                    left_on='Peptide',
                    right_on='peptide',
                    how='left',
                    suffixes=('', '_filtered')
)
# peptide column
if 'peptide' in enhanced_df.columns:
    enhanced_df = enhanced_df.drop(columns=['peptide'])
else:
    enhanced_df = merged_df
    print(f" Info: No MHC info to merge (only marking peptides of interest)")
    # column column, information first
    priority_cols = [
        'Peptide', 'CDR3', 'Score', # PanPep column
        'is_filtered_peptide', # new: for peptide
        'MHC', 'Affinity(nM)', 'BindLevel', # MHCinformation( with)
        'total_count', 'total_frequency', # TRUST4 information
        'n_variants', 'primary_variant_count',
        'CDR3nt_with_counts',
        'V', 'D', 'J', 'C', 'cid', 'cid_full_length',
        'has_annotation_conflict'
    ]
    existing_priority_cols = [c for c in priority_cols if c in enhanced_df.columns]
    other_cols = [c for c in enhanced_df.columns if c not in existing_priority_cols]
    enhanced_df = enhanced_df[existing_priority_cols + other_cols]
    # statistics
    n_filtered = enhanced_df['is_filtered_peptide'].sum()
    print(f" Summary: Final statistics:")
    print(f" OK Filtered peptides: {n_filtered} ({n_filtered/original_len*100:.1f}%)")
    if 'MHC' in enhanced_df.columns:
        n_with_mhc = enhanced_df['MHC'].notna().sum()
        print(f" OK With MHC info: {n_with_mhc} ({n_with_mhc/original_len*100:.1f}%)")
        # peptidein, statisticsMHCandBindLevelinformation
        filtered_df = enhanced_df[enhanced_df['is_filtered_peptide']]
        if len(filtered_df) > 0:
            n_filtered_with_mhc = filtered_df['MHC'].notna().sum()
            print(f" Info: Filtered peptides with MHC: {n_filtered_with_mhc}/{n_filtered}")
            if 'BindLevel' in filtered_df.columns:
                bind_level_counts = filtered_df['BindLevel'].value_counts(dropna=False)
                print(f" Target: BindLevel distribution (filtered peptides only):")
                for level, count in bind_level_counts.items():
                    if pd.isna(level):
                        print(f" Empty/No bind: {count}")
                    else:
                        print(f" {level}: {count}")
                        # saveresults
                        os.makedirs(os.path.dirname(output_file), exist_ok=True)
                        enhanced_df.to_csv(output_file, index=False)
                        print(f" OK Saved: {os.path.basename(output_file)}")
                        return True
                        except Exception as e:
                            print(f" Failed Error processing: {e}")
                            import traceback
                            traceback.print_exc()
                            return False

def process_single_sample(project, group, sample, mhc_class, base_results_dir):
    """
    process samples MHC class
    Summary:
    project: 'CRC' or 'CRC_MS'
    group: 'R' or 'T'
    sample: sample
    mhc_class: 'I' or 'II'
    base_results_dir: results directory
    """
    print(f"\n{'-'*60}")
    print(f"Sample: {sample} | Class {mhc_class}")
    print(f"{'-'*60}")
    # buildfilepath
    # Filteredfilepath
    filtered_file = os.path.join(
        base_results_dir, project,
        '06.antigen', '06.2.MTR_peptides', 'R_fully_contained',
        f'{sample}_class{mhc_class}_filtered.tsv'
)
# Mergedfilepath - filenameformat
if mhc_class == 'I':
    mhc_suffix = 'p_I'
else:
    mhc_suffix = 'p_II'
    # filenameformat
    merged_file_v2 = os.path.join(
        base_results_dir, project,
        '07.TRUST4', 'PanPep_merged',
        f'{project}.{group}.{sample}.{mhc_suffix}.panpep_zero_shot_out_v2_merged.csv'
)
merged_file_v1 = os.path.join(
    base_results_dir, project,
    '07.TRUST4', 'PanPep_merged',
    f'{project}.{group}.{sample}.{mhc_suffix}.panpep_zero_shot_out_merged.csv'
)
# does not existfile
if os.path.exists(merged_file_v2):
    merged_file = merged_file_v2
elif os.path.exists(merged_file_v1):
    merged_file = merged_file_v1
else:
    print(f" Failed Merged file not found (tried both v1 and v2 formats)")
    return False
    # Output filespath - filename
    output_dir = os.path.join(
        base_results_dir, project,
        '07.TRUST4', 'Final_Results'
)
# filename: . .sample. .final.csv
output_file = os.path.join(output_dir, f'{project}.{group}.{sample}.{mhc_suffix}.final.csv')
print(f"Folder: Filtered file: {os.path.basename(filtered_file)}")
print(f"Folder: Merged file: {os.path.basename(merged_file)}")
# readfiltered peptideinformation
filtered_info, filtered_peptides = load_filtered_peptides(filtered_file)
# MHCinformationtomergedfile
success = add_mhc_info_to_merged(merged_file, filtered_info, filtered_peptides, output_file)
return success

def process_all_samples(base_results_dir):
    """
    process all samples
    """
    projects = {
        'CRC': {'group': 'R'},
        'CRC_MS': {'group': 'T'}
    }
    success_count = 0
    total_count = 0
    for project, config in projects.items():
        print(f"\n{'='*60}")
        print(f"Project: PROJECT: {project}")
        print(f"{'='*60}")
        group = config['group']
        # allmergedfile
        merged_dir = os.path.join(
            base_results_dir, project,
            '07.TRUST4', 'PanPep_merged'
)
if not os.path.exists(merged_dir):
    print(f"Failed Merged directory not found: {merged_dir}")
    continue
    merged_files = sorted([f for f in os.listdir(merged_dir)
        if f.endswith('_merged.csv') and ('panpep_zero_shot_out_v2_merged.csv' in f or 'panpep_zero_shot_out_merged.csv' in f)])
    print(f"Files: Found {len(merged_files)} merged files")
    for merged_file in merged_files:
        # filename - format
        if '_v2_merged.csv' in merged_file:
            parts = merged_file.replace('.panpep_zero_shot_out_v2_merged.csv', '').split('.')
        else:
            parts = merged_file.replace('.panpep_zero_shot_out_merged.csv', '').split('.')
            if len(parts) >= 4:
                sample = parts[2]
                mhc_suffix = parts[3] # p_I or p_II
                # forclass I or class II
                if mhc_suffix == 'p_I':
                    mhc_class = 'I'
                elif mhc_suffix == 'p_II':
                    mhc_class = 'II'
                else:
                    print(f"Warning Unknown MHC suffix: {mhc_suffix}")
                    continue
                total_count += 1
                if process_single_sample(project, group, sample, mhc_class, base_results_dir):
                    success_count += 1
                else:
                    print(f"Warning Cannot parse filename: {merged_file}")
                    return success_count, total_count

def generate_summary_report(base_results_dir):
    """
    generateafterdata report
    """
    projects = ['CRC', 'CRC_MS']
    all_stats = []
    for project in projects:
        final_dir = os.path.join(
            base_results_dir, project,
            '07.TRUST4', 'Final_Results'
)
if not os.path.exists(final_dir):
    continue
    for file in os.listdir(final_dir):
        if not file.endswith('.final.csv'):
            continue
        filepath = os.path.join(final_dir, file)
        try:
            df = pd.read_csv(filepath)
            # filename: PROJECT.GROUP.SAMPLE.MHC.final.csv
            parts = file.replace('.final.csv', '').split('.')
            if len(parts) < 4:
                continue
            sample = parts[2]
            mhc_type = 'Class I' if 'p_I' in file else 'Class II'
            total = len(df)
            # statistics peptide
            if 'is_filtered_peptide' in df.columns:
                n_filtered = df['is_filtered_peptide'].sum()
                filtered_pct = f"{n_filtered/total*100:.1f}%"
            else:
                n_filtered = 0
                filtered_pct = "N/A"
                # statisticsMHCinformation
                if 'MHC' in df.columns:
                    with_mhc = df['MHC'].notna().sum()
                    mhc_pct = f"{with_mhc/total*100:.1f}%"
                    # peptideinwithMHCinformation
                    if n_filtered > 0:
                        filtered_with_mhc = df[df['is_filtered_peptide'] & df['MHC'].notna()].shape[0]
                    else:
                        filtered_with_mhc = 0
                else:
                    with_mhc = 0
                    mhc_pct = "N/A"
                    filtered_with_mhc = 0
                    # statisticsBindLevel( peptidein)
                    bind_stats = {}
                    if n_filtered > 0 and 'BindLevel' in df.columns:
                        filtered_df = df[df['is_filtered_peptide']]
                        for level in ['<= SB', '<= WB']:
                            count = (filtered_df['BindLevel'] == level).sum()
                            if count > 0:
                                bind_stats[level] = count
                                # BindLevel(No bindor )
                                no_bind = filtered_df['BindLevel'].isna().sum()
                                if no_bind > 0:
                                    bind_stats['No/Empty bind'] = no_bind
                                    stats = {
                                        'Project': project,
                                        'Sample': sample,
                                        'MHC_Class': mhc_type,
                                        'Total_Records': total,
                                        'Filtered_Peptides': n_filtered,
                                        'Filtered_Pct': filtered_pct,
                                        'Filtered_With_MHC': filtered_with_mhc
                                    }
                                    # BindLevelstatistics
                                    stats.update(bind_stats)
                                    all_stats.append(stats)
        except Exception as e:
            print(f"Error reading {file}: {e}")
            return pd.DataFrame(all_stats)

if __name__ == '__main__':
    BASE_RESULTS_DIR = '/path/to/project/results_V3/special'
    print("\n" + "="*60)
    print("Tool: Add MHC Information & Mark Filtered Peptides")
    print("="*60)
    print(f"Folder: Results directory: {BASE_RESULTS_DIR}")
    print("="*60)
    print("\nTarget: Tasks:")
    print(" - Extract MHC, Affinity, BindLevel from filtered files")
    print(" - Mark ALL filtered peptides (even with empty BindLevel)")
    print(" - Add to merged PanPep-TRUST4 results")
    print(" - Simplify output file names")
    print("="*60)
    # process all samples
    success, total = process_all_samples(BASE_RESULTS_DIR)
    # generatereport
    print("\n" + "="*60)
    print("Statistics: Generating Summary Report")
    print("="*60)
    summary_df = generate_summary_report(BASE_RESULTS_DIR)
    if len(summary_df) > 0:
        summary_file = os.path.join(BASE_RESULTS_DIR, 'final_results_summary.csv')
        summary_df.to_csv(summary_file, index=False)
        print(f"\n{summary_df.to_string(index=False)}")
        print(f"\nOK Summary saved to: {summary_file}")
        print("\n" + "="*60)
        print(f"Completed: Completed! Processed {success}/{total} files successfully")
        print("="*60)
        print("\nFiles: Output directory:")
        print(" */07.TRUST4/Final_Results/")
        print("\nStatistics: File naming: PROJECT.GROUP.SAMPLE.MHC.final.csv")
        print("\nTarget: New column 'is_filtered_peptide':")
        print(" True = peptide of interest (from filtered files)")
        print(" False = all other peptides")
        print("\nInfo: Note: Some filtered peptides may have empty BindLevel")
        print(" (meaning No bind), but they're still marked as")
        print(" 'is_filtered_peptide=True'")
        print("="*60)